# Experiment 3.0.1 — Single-$\tau_{syn}$ × objective comparison

## Question
At a fixed 3-layer feature SNN, which homogeneous synaptic time scale is best, and does the answer depend on the training objective?

Backbone: `30 -> 128 -> 128 -> 64`, no recurrence, no bias in SNN linear layers. L1/L2/L3 use the same `shift_syn` within one run. Sweep shifts `[2,3,4,5,6,7]`.

Objectives:
- `timestep_ce`: shared `64 -> 12` classifier at every valid timestep; CE averaged over valid timesteps.
- `relative10_sequence_ce`: valid gesture divided into 10 relative bins; L3 spike counts are flattened and classified once per segment.
- `fixed250_sequence_ce`: valid-masked 250 ms bins (16 samples at 64 Hz); L3 spike counts are flattened and classified once per segment. SNN state is not reset at bin boundaries.

Grid: `6 shifts × 3 objectives × 3 seeds = 54 runs`, seeds `(11,23,101)`, split seed `12345`.

## Execution model
Formal training is performed by the Unity Slurm job array `scripts/bash_script/SNN_Bash/run_exp_3_0_1_cpu_array.bash`. Each array task owns one `(shift, objective, seed)` run and writes only its own checkpoint. After all 54 runs finish, `scripts/experiment_3_0_1/02_finalize_experiment.py` combines checkpoints into the CSV artifacts consumed here. This notebook performs analysis and plotting only.

## 1. Load finalized artifacts

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

def find_repo_root(start=None):
    start=(start or Path.cwd()).resolve()
    for p in (start,*start.parents):
        if (p/'snn').is_dir() and (p/'notebooks').is_dir(): return p
    raise FileNotFoundError('writingRing repo root not found')

REPO_ROOT=find_repo_root()
if str(REPO_ROOT) not in sys.path: sys.path.insert(0,str(REPO_ROOT))
from scripts.experiment_3_0_1_single_tau_objectives import EXPERIMENT_ID, EXPECTED_RUNS, SHIFTS, OBJECTIVES, SEEDS, WIDTHS

RESULTS_DIR=REPO_ROOT/'notebooks/artifacts'/EXPERIMENT_ID
RESULTS_PATH=RESULTS_DIR/'experiment_3_0_1_results.csv'
HISTORY_PATH=RESULTS_DIR/'experiment_3_0_1_history.csv'
SUMMARY_PATH=RESULTS_DIR/'experiment_3_0_1_summary.csv'
missing=[p for p in (RESULTS_PATH,HISTORY_PATH,SUMMARY_PATH) if not p.exists()]
if missing:
    raise FileNotFoundError('Finalized Experiment 3.0.1 artifacts are missing. Run the Slurm array and finalizer first:\n' + '\n'.join(map(str,missing)))
RESULTS=pd.read_csv(RESULTS_PATH)
HISTORY=pd.read_csv(HISTORY_PATH)
SUMMARY=pd.read_csv(SUMMARY_PATH)
if len(RESULTS)!=EXPECTED_RUNS: raise ValueError(f'Expected {EXPECTED_RUNS} runs, found {len(RESULTS)}')
print('Repository root:',REPO_ROOT)
print('Experiment:',EXPERIMENT_ID)
print('Widths:',WIDTHS)
print('Shifts:',SHIFTS)
print('Objectives:',OBJECTIVES)
print('Seeds:',SEEDS)
print('Completed runs:',len(RESULTS),'/',EXPECTED_RUNS)
display(SUMMARY)

## 2. Per-run metrics

In [ ]:
cols=['objective','shift','tau_syn_ms','seed','best_epoch','train_balanced_accuracy','val_balanced_accuracy','test_balanced_accuracy','test_macro_f1','test_accuracy','probe_test_balanced_accuracy','train_test_ba_gap']
display(RESULTS[cols].sort_values(['objective','shift','seed']))

## 3. Native test balanced accuracy vs shift

In [ ]:
fig,ax=plt.subplots(figsize=(8,5))
for obj in OBJECTIVES:
    p=SUMMARY[SUMMARY.objective==obj].sort_values('shift')
    ax.errorbar(p['shift'],p['mean_test_balanced_accuracy'],yerr=p['sd_test_balanced_accuracy'],marker='o',capsize=3,label=obj)
ax.set(xlabel='homogeneous shift_syn in L1/L2/L3',ylabel='test balanced accuracy',xticks=SHIFTS)
ax.grid(alpha=.25); ax.legend(); plt.show()

## 4. Common fixed-250-ms frozen linear-probe BA

In [ ]:
fig,ax=plt.subplots(figsize=(8,5))
for obj in OBJECTIVES:
    p=SUMMARY[SUMMARY.objective==obj].sort_values('shift')
    ax.errorbar(p['shift'],p['mean_probe_test_balanced_accuracy'],yerr=p['sd_probe_test_balanced_accuracy'],marker='o',capsize=3,label=obj)
ax.set(xlabel='homogeneous shift_syn in L1/L2/L3',ylabel='common fixed250 probe test BA',xticks=SHIFTS)
ax.grid(alpha=.25); ax.legend(); plt.show()

## 5. Validation BA vs epoch

In [ ]:
M=HISTORY.groupby(['objective','shift','epoch'],as_index=False).val_balanced_accuracy.mean()
for obj in OBJECTIVES:
    fig,ax=plt.subplots(figsize=(8,5)); p=M[M.objective==obj]
    for shift in SHIFTS:
        q=p[p['shift']==shift]; ax.plot(q.epoch,q.val_balanced_accuracy,label=f'shift {shift}')
    ax.set(title=obj,xlabel='epoch',ylabel='mean validation BA'); ax.grid(alpha=.25); ax.legend(ncol=2); plt.show()

## 6. Firing-rate diagnostics

In [ ]:
fr_cols=['objective','shift','seed','test_l1_firing_rate','test_l2_firing_rate','test_l3_firing_rate','zero_l1_firing_rate','zero_l2_firing_rate','zero_l3_firing_rate']
display(RESULTS[fr_cols])
for obj in OBJECTIVES:
    fig,ax=plt.subplots(figsize=(8,5))
    p=RESULTS[RESULTS.objective==obj].groupby('shift',as_index=False)[['test_l1_firing_rate','test_l2_firing_rate','test_l3_firing_rate']].mean()
    for col,label in [('test_l1_firing_rate','L1'),('test_l2_firing_rate','L2'),('test_l3_firing_rate','L3')]: ax.plot(p['shift'],p[col],marker='o',label=label)
    ax.set(title=obj,xlabel='shift_syn',ylabel='valid spikes / neuron / timestep',xticks=SHIFTS); ax.grid(alpha=.25); ax.legend(); plt.show()

## 7. Final rankings

In [ ]:
native=RESULTS.groupby(['objective','shift']).test_balanced_accuracy.agg(['mean','std']).sort_values('mean',ascending=False)
probe=RESULTS.groupby(['objective','shift']).probe_test_balanced_accuracy.agg(['mean','std']).sort_values('mean',ascending=False)
print('Native head ranking'); display(native)
print('Common frozen-probe ranking'); display(probe)

## Interpretation
Use the three-seed mean and SD rather than any single run. First determine whether a consistent single-$\tau$ optimum exists. Then test whether that optimum changes across supervision objectives. Compare native-head BA with the common fixed250 frozen probe: agreement supports a representation-level effect, while disagreement suggests the training/readout head contributes materially. Inspect firing rates and train–test gap before selecting the single-$\tau$ baseline for the subsequent multi-$\tau$ architecture study.